# Thai Checkers Neural Network V2 - Training on Colab GPU

This notebook trains the Thai Checkers NN V2 on Google Colab GPU.

## Setup:
1. Upload this notebook to Google Colab
2. Change runtime to GPU (Runtime → Change runtime type → GPU)
3. Upload training data from `.tmp/training_data_with_features/`
4. Run all cells

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Upload training data
# Option 1: Upload manually via Colab file browser
# Option 2: Mount Google Drive and copy from there
from google.colab import drive
drive.mount('/content/drive')

# Copy training data from Drive
!mkdir -p /content/.tmp/training_data_with_features
!cp /content/drive/MyDrive/thai_checkers_nn_training/training_data/*.json /content/.tmp/training_data_with_features/

In [ ]:
# Check uploaded data
import os
data_files = [f for f in os.listdir('/content/.tmp/training_data_with_features') if f.endswith('.json')]
print(f'Found {len(data_files)} data files:')
for f in sorted(data_files)[:10]:
    size = os.path.getsize(f'/content/.tmp/training_data_with_features/{f}') / 1024
    print(f'  {f}: {size:.1f} KB')

## Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, hidden: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )

    def forward(self, x):
        return F.relu(x + self.net(x))

class ThaiCheckersNetV2(nn.Module):
    def __init__(self, hidden: int = 384, n_res: int = 12, dropout: float = 0.1):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Linear(320, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Trunk
        self.trunk = nn.Sequential(
            *[ResBlock(hidden, dropout=dropout) for _ in range(n_res)]
        )

        # Policy head
        self.policy_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1024),
        )

        # Value head
        self.value_head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, features):
        x = self.stem(features)
        x = self.trunk(x)
        policy_logits = self.policy_head(x)
        value = self.value_head(x)
        return policy_logits, value

# Test model
model = ThaiCheckersNetV2()
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## Dataset and DataLoader

In [ ]:
import json
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class ThaiCheckersDataset(Dataset):
    def __init__(self, data_files, augment=True):
        self.examples = []
        self.augment = augment
        for file_path in data_files:
            print(f'Loading {file_path}...')
            with open(file_path, 'r') as f:
                data = json.load(f)
                self.examples.extend(data)
        print(f'Total: {len(self.examples)} examples (augment={augment})')

    def __len__(self):
        return len(self.examples)

    def _flip_features(self, features):
        """Flip board features for P1/P2 side symmetry augmentation"""
        flipped = features[:]  # Make a copy

        # Flip positional features [0-127]: my_men, my_kings, op_men, op_kings
        # Swap my pieces <-> opponent pieces
        for layer in range(4):
            for sq in range(32):
                # Swap layers: 0↔2 (my_men↔op_men), 1↔3 (my_kings↔op_kings)
                target_layer = 2 + layer if layer < 2 else layer - 2
                flipped[target_layer * 32 + (31 - sq)] = features[layer * 32 + sq]

        # Flip mobility features [128-191]: my_mobility, op_mobility
        for layer in range(2):
            for sq in range(32):
                # Swap layers: 0↔1 (my_mobility↔op_mobility)
                target_layer = 1 - layer
                flipped[128 + target_layer * 32 + (31 - sq)] = features[128 + layer * 32 + sq]

        # Flip threat maps [192-255]: my_threats, op_threats
        for layer in range(2):
            for sq in range(32):
                # Swap layers: 0↔1 (my_threats↔op_threats)
                target_layer = 1 - layer
                flipped[192 + target_layer * 32 + (31 - sq)] = features[192 + layer * 32 + sq]

        # Flip hanging flags [256-287]: hanging pieces
        for sq in range(32):
            flipped[256 + (31 - sq)] = features[256 + sq]

        # Flip distance to promotion [288-319]: promotion distance
        for sq in range(32):
            # Distance to promotion also inverts (distance from bottom → distance from top)
            # 7 - distance because board height is 8
            original_dist = features[288 + sq]
            flipped[288 + (31 - sq)] = 7 - original_dist if original_dist > 0 else 0

        return flipped

    def _flip_move(self, from_sq, to_sq):
        """Flip move indices for P1/P2 side symmetry"""
        return (31 - from_sq, 31 - to_sq)

    def __getitem__(self, idx):
        example = self.examples[idx]
        features = example['features'][:]  # Make a copy
        best_move = example['bestMove']
        value = example['positionValue']

        # Data augmentation: 50% chance to flip P1/P2 (CRITICAL FIX!)
        if self.augment and np.random.rand() < 0.5:
            features = self._flip_features(features)
            from_sq, to_sq = self._flip_move(best_move['from'], best_move['to'])
            move_idx = from_sq * 32 + to_sq
            value = -value  # Flip value for opponent's perspective
        else:
            move_idx = best_move['from'] * 32 + best_move['to']

        return {
            'features': torch.tensor(features, dtype=torch.float32),
            'move_idx': move_idx,
            'value': torch.tensor([value], dtype=torch.float32),
        }

# Load data
data_dir = Path('/content/.tmp/training_data_with_features')
train_files = list(data_dir.glob('chunk_*.json'))

print(f'Found {len(train_files)} chunk files')

# For small datasets (< 5 chunks), use all data for both train and val
if len(train_files) < 5:
    print(f'Small dataset detected - using all data for train & val')
    train_dataset = ThaiCheckersDataset([str(f) for f in train_files], augment=True)
    val_dataset = ThaiCheckersDataset([str(f) for f in train_files], augment=False)  # No augment for validation
else:
    # 80/20 split for larger datasets
    split_idx = int(len(train_files) * 0.8)
    train_dataset = ThaiCheckersDataset([str(f) for f in train_files[:split_idx]], augment=True)
    val_dataset = ThaiCheckersDataset([str(f) for f in train_files[split_idx:]], augment=False)

# Create dataloaders
BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} examples, {len(train_loader)} batches')
print(f'Val: {len(val_dataset)} examples, {len(val_loader)} batches')

## Training Loop

In [ ]:
import torch.optim as optim

# Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LEARNING_RATE = 0.001
EPOCHS = 100

model = ThaiCheckersNetV2().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

print(f'Training on: {DEVICE}')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    total_policy_loss = 0
    total_value_loss = 0
    correct = 0
    total = 0

    for batch in dataloader:
        features = batch['features'].to(device)
        move_idx = batch['move_idx'].to(device)
        value_target = batch['value'].to(device)

        policy_logits, value_pred = model(features)

        policy_loss = nn.functional.cross_entropy(policy_logits, move_idx)
        value_loss = nn.functional.mse_loss(value_pred, value_target)
        loss = policy_loss + value_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_policy_loss += policy_loss.item()
        total_value_loss += value_loss.item()

        pred_moves = torch.argmax(policy_logits, dim=1)
        correct += (pred_moves == move_idx).sum().item()
        total += len(move_idx)

    return {
        'loss': total_loss / len(dataloader),
        'policy_loss': total_policy_loss / len(dataloader),
        'value_loss': total_value_loss / len(dataloader),
        'accuracy': correct / total,
    }

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_policy_loss = 0
    total_value_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            features = batch['features'].to(device)
            move_idx = batch['move_idx'].to(device)
            value_target = batch['value'].to(device)

            policy_logits, value_pred = model(features)

            policy_loss = nn.functional.cross_entropy(policy_logits, move_idx)
            value_loss = nn.functional.mse_loss(value_pred, value_target)
            loss = policy_loss + value_loss

            total_loss += loss.item()
            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()

            pred_moves = torch.argmax(policy_logits, dim=1)
            correct += (pred_moves == move_idx).sum().item()
            total += len(move_idx)

    return {
        'loss': total_loss / len(dataloader),
        'policy_loss': total_policy_loss / len(dataloader),
        'value_loss': total_value_loss / len(dataloader),
        'accuracy': correct / total,
    }

In [ ]:
# Training loop
import os
os.makedirs('checkpoints', exist_ok=True)

best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoch {epoch}/{EPOCHS}')

    # Train
    train_stats = train_epoch(model, train_loader, optimizer, DEVICE)
    print(f'  Train - Loss: {train_stats["loss"]:.4f}, '
          f'Policy: {train_stats["policy_loss"]:.4f}, '
          f'Value: {train_stats["value_loss"]:.4f}, '
          f'Acc: {train_stats["accuracy"]:.2%}')

    # Validate
    val_stats = validate(model, val_loader, DEVICE)
    print(f'  Val   - Loss: {val_stats["loss"]:.4f}, '
          f'Policy: {val_stats["policy_loss"]:.4f}, '
          f'Value: {val_stats["value_loss"]:.4f}, '
          f'Acc: {val_stats["accuracy"]:.2%}')

    # Learning rate scheduling
    scheduler.step(val_stats['accuracy'])

    # Save history
    history['train_loss'].append(train_stats['loss'])
    history['train_acc'].append(train_stats['accuracy'])
    history['val_loss'].append(val_stats['loss'])
    history['val_acc'].append(val_stats['accuracy'])

    # Save best model
    if val_stats['accuracy'] > best_val_acc:
        best_val_acc = val_stats['accuracy']
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')
        print(f'  ✅ New best model! Accuracy: {best_val_acc:.2%}')

    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_stats': train_stats,
            'val_stats': val_stats,
            'history': history,
        }, f'checkpoints/checkpoint_epoch_{epoch}.pt')

print('\n' + '='*80)
print(f'Training complete! Best validation accuracy: {best_val_acc:.2%}')
print('='*80)

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Export ONNX Model

In [ ]:
# Install onnxscript for ONNX export (required for PyTorch 2.x)
!pip install -q onnxscript onnx

In [ ]:
# Load best model
model.load_state_dict(torch.load('checkpoints/best_model.pt'))
model.eval()

# Convert model to CPU for export
model_cpu = model.cpu()
dummy_input_cpu = torch.randn(1, 320)

# Export ONNX first (may create external data file)
torch.onnx.export(
    model_cpu,
    dummy_input_cpu,
    'checkpoints/thai_checkers_v2_temp.onnx',
    input_names=['features'],
    output_names=['policy_logits', 'value'],
    dynamic_axes={'features': {0: 'batch'}},
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
)

# Convert to single-file ONNX using onnx library
import onnx

# Load the model with external data
onnx_model = onnx.load('checkpoints/thai_checkers_v2_temp.onnx')

# Save as single file (embed all data)
onnx.save(
    onnx_model,
    'checkpoints/thai_checkers_v2.onnx',
    save_as_external_data=False  # Force single file
)

# Clean up temp files
import os
if os.path.exists('checkpoints/thai_checkers_v2_temp.onnx'):
    os.remove('checkpoints/thai_checkers_v2_temp.onnx')
if os.path.exists('checkpoints/thai_checkers_v2_temp.onnx.data'):
    os.remove('checkpoints/thai_checkers_v2_temp.onnx.data')

print('✅ ONNX model exported to checkpoints/thai_checkers_v2.onnx')

# Check file size
size_mb = os.path.getsize('checkpoints/thai_checkers_v2.onnx') / (1024 * 1024)
print(f'Model size: {size_mb:.2f} MB')

# Verify it's a single file (no external data)
if os.path.exists('checkpoints/thai_checkers_v2.onnx.data'):
    print('⚠️  WARNING: External data file created - model is not single-file')
    print('Trying to combine files...')
    
    # If still has external data, use convert script
    !python /content/drive/MyDrive/thai_checkers_nn_training/convert_onnx_single_file.py checkpoints/thai_checkers_v2.onnx checkpoints/thai_checkers_v2_single.onnx
    
    # Replace with single file version
    if os.path.exists('checkpoints/thai_checkers_v2_single.onnx'):
        os.remove('checkpoints/thai_checkers_v2.onnx')
        os.rename('checkpoints/thai_checkers_v2_single.onnx', 'checkpoints/thai_checkers_v2.onnx')
        if os.path.exists('checkpoints/thai_checkers_v2.onnx.data'):
            os.remove('checkpoints/thai_checkers_v2.onnx.data')
        print('✅ Converted to single-file ONNX successfully')
else:
    print('✅ Single-file ONNX model (no external data)')

## Download Results

Download the following files to your computer:
- `checkpoints/best_model.pt` - Best PyTorch checkpoint
- `checkpoints/thai_checkers_v2.onnx` - ONNX model for deployment
- `training_curves.png` - Training visualization

In [ ]:
# Download files
from google.colab import files

files.download('checkpoints/best_model.pt')
files.download('checkpoints/thai_checkers_v2.onnx')
files.download('training_curves.png')